# 🧠 Final Project 3 - MBDA
## CNN for EMNIST Alphabet Classification (A–Z)
**Course:** Multimodal Biomedical Data Analysis (MBDA)
**Dataset:** EMNIST Letters (26 classes: A–Z)
**Framework:** PyTorch + GPU (CUDA)
**Visualization:** Seaborn + Matplotlib (Interactive via PyQt6)

---
## Differences from the lecturer's MNIST example:
| Aspect | MNIST (lecturer) | EMNIST Letters (us) |
|---|---|---|
| Class | 10 (digits 0–9) | 26 (A–Z) |
| Training Data | 60,000 | 88,800 |
| Test Data | 10,000 | 14,800 |
| Conv layers | 2 | 3 + BatchNorm |
| FC layers | 3 (84 neurons) | 3 (512 neurons) |
| Devices | CPU | **GPU (CUDA)** |
| Batch sizes | 10 | **128** |

## Cell 1 – Import Library

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from tqdm.auto import tqdm
import time
import os
import sys
import warnings
warnings.filterwarnings('ignore')


# Visualization Settings
# MATPLOTLIB BACKEND OPTIONS:
# %matplotlib inline → static display inside Jupyter (default)
# %matplotlib widget → interactive inline (pip install ipympl)
# %matplotlib qt6 → separate Qt6 window (requires PyQt6)

%matplotlib inline
%gui qt6

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.dpi'      : 110,
    'figure.figsize'  : (12, 6),
    'font.size'       : 11,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'axes.titleweight': 'bold',
})

print('Library successfully imported.')

Library successfully imported.


## Cell 2 – GPU / CPU Configuration

In [19]:
# Device Detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"{'='*55}")
print(f"  Device yang digunakan : {device}")
print(f"{'='*55}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Model             : {props.name}")
    print(f"  VRAM Total            : {props.total_memory / 1024**3:.1f} GB")
    print(f"  CUDA Capability       : {props.major}.{props.minor}")
    print(f"  PyTorch CUDA Build    : {torch.version.cuda}")
    print(f"  cuDNN Enabled         : {torch.backends.cudnn.enabled}")
    torch.backends.cudnn.benchmark = True   # auto-tune konvolusi untuk performa terbaik
    print(f"  cuDNN Benchmark Mode  : ON (activated for optimal performance)")
else:
    print("  ⚠️  GPU not detected. Please install PyTorch CUDA version.")
    print("  Install: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124")

print(f"{'='*55}")

# Global Hyperparameters
BATCH_SIZE   = 128   # optimal for RTX 4050 (6 GB VRAM)
EPOCHS       = 15
LEARNING_RATE= 0.001
NUM_CLASSES  = 26    # A–Z
NUM_WORKERS  = 4     # threads for DataLoader (0 = main thread)
SEED         = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"\n  Batch size   : {BATCH_SIZE}")
print(f"  Epochs       : {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Num classes  : {NUM_CLASSES} (A–Z)")

NameError: name 'torch' is not defined

## Cell 3 – Load EMNIST Letters Dataset

> **Important note:** The EMNIST Letters dataset has two quirks:
> 1. **Labels start at 1** (not 0). Need `target_transform` to shift them to 0–25.
> 2. **The image is transposed** compared to MNIST. Need `transforms.Lambda` to correct the orientation.

In [18]:
# Transform
from utils import fix_orientation, adjust_label

# Mean/std value of EMNIST dataset (same as MNIST because distribution is similar)
_mean = (0.1307,)
_std  = (0.3081,)

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
   # ▶ Orientation fix: EMNIST stored transposed vs MNIST
    #transforms.Lambda(lambda img: img.permute(0, 2, 1)),
    transforms.Lambda(fix_orientation),
   # ▶ Light augmentation for regularization
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
    #transforms.Lambda(lambda img: img.permute(0, 2, 1)),   # fix orientation for EMNIST
    transforms.Lambda(fix_orientation),
])

# Labels 1–26 → 0–25 to be compatible with CrossEntropyLoss
#target_transform = transforms.Lambda(lambda y: y - 1)

# Download and Load
train_data = datasets.EMNIST(
    root='./data', split='letters', train=True,
    download=True, transform=transform_train, target_transform=adjust_label
)
test_data = datasets.EMNIST(
    root='./data', split='letters', train=False,
    download=True, transform=transform_test, target_transform=adjust_label
)

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=True
)
test_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=True
)

# Alphabet Mapping
ALPHABET = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

print(f"Training samples : {len(train_data):,}")
print(f"Test samples     : {len(test_data):,}")
print(f"Jumlah kelas     : {NUM_CLASSES}")
print(f"Kelas            : {ALPHABET}")
print(f"\nBatches per epoch (train): {len(train_loader)}")
print(f"Batches per epoch (test) : {len(test_loader)}")
print(f"Ukuran gambar    : {train_data[0][0].shape}")

NameError: name 'transforms' is not defined

## Cell 4 – Dataset Exploration: Sample Images per Letter

In [17]:
# Index per class
targets_np = np.array(train_data.targets) - 1   # shift label 1-26 → 0-25

fig, axes = plt.subplots(4, 7, figsize=(16, 9))
fig.suptitle('EMNIST Letters: Example Images per Class (After Orientation Fix)',
             fontsize=14, fontweight='bold', y=1.02)

for i, letter in enumerate(ALPHABET):
    row, col = i // 7, i % 7
    idx = np.where(targets_np == i)[0][5]     # take 5th sample of each class
    img, label = train_data[idx]
    
    axes[row, col].imshow(img.squeeze().numpy(), cmap='gray', interpolation='bilinear')
    axes[row, col].set_title(letter, fontsize=13, fontweight='bold', color='#2c3e50')
    axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    for spine in axes[row, col].spines.values():
        spine.set_edgecolor('#bdc3c7'); spine.set_linewidth(0.8)

# SHide unused subplots (total 28, used 26)
axes[3, 5].set_visible(False)
axes[3, 6].set_visible(False)

plt.tight_layout()
plt.savefig('result/plot_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: result/plot_sample_images.png")

NameError: name 'np' is not defined

## Cell 5 – Class Distribution (Bar Chart)

In [16]:
# Calculate class distribution
train_counts = [(targets_np == i).sum() for i in range(NUM_CLASSES)]

palette = sns.color_palette('husl', NUM_CLASSES)

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(ALPHABET, train_counts, color=palette, edgecolor='white', linewidth=0.6)

# Add count labels on top of bars
for bar, count in zip(bars, train_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{count:,}', ha='center', va='bottom', fontsize=7, color='#2c3e50')

ax.axhline(np.mean(train_counts), color='crimson', linestyle='--', linewidth=1.5,
           label=f'Average: {np.mean(train_counts):.0f}')
ax.set_xlabel('Alphabet Letters')
ax.set_ylabel('Number of Training Samples')
ax.set_title('Class Distribution in EMNIST Letters Dataset')
ax.legend()
ax.set_ylim(0, max(train_counts) * 1.15)

plt.tight_layout()
plt.savefig('result/plot_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Most frequent class : {ALPHABET[np.argmax(train_counts)]} ({max(train_counts):,} samples)")
print(f"Least frequent class: {ALPHABET[np.argmin(train_counts)]} ({min(train_counts):,} samples)")

NameError: name 'NUM_CLASSES' is not defined

## Cell 6 – CNN Architecture

Architecture Comparison:
```
Lecturer's MNIST:          My EMNIST:
───────────────────   ─────────────────────────────────────────
Conv(1→6,  k=3)       ConvBlock1: Conv(1→32,  k=3, pad=1) + BN + ReLU + Pool
MaxPool               ConvBlock2: Conv(32→64, k=3, pad=1) + BN + ReLU + Pool
Conv(6→16, k=3)       ConvBlock3: Conv(64→128,k=3, pad=1) + BN + ReLU + Pool
MaxPool               ─────────────────────────────────────────
FC: 400→120           FC1: 1152 → 512 + ReLU + Dropout(0.5)
FC: 120→84            FC2: 512  → 256 + ReLU + Dropout(0.3)
FC: 84→10             FC3: 256  → 26  (output)
```
**Why more complex?** 26 classes are much harder to distinguish (many similar letters: C/G, I/L/J, etc.)

**BatchNorm** → stabilizes training, speeds up convergence
**Dropout** → regularization, prevents overfitting
**Padding=1** → maintains spatial size after convolution

In [15]:
class EMNISTConvNet(nn.Module):
    """
    3-layer CNN for EMNIST Letters classification (A–Z, 26 classes)

    Dimensional flow (input: 1×28×28):
      ConvBlock1 → Pool : 32×28×28 → 32×14×14
      ConvBlock2 → Pool : 64×14×14 → 64×7×7
      ConvBlock3 → Pool : 128×7×7  → 128×3×3
      Flatten           : 1152
      FC1               : 1152 → 512
      FC2               : 512  → 256
      FC3 (output)      : 256  → 26
    """
    def __init__(self, num_classes=26, drop1=0.5, drop2=0.3):
        super().__init__()

        # Convolutional Blocks
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 28×28 → 14×14
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 14×14 → 7×7
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 7×7 → 3×3
        )

        # Fully Connected Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(drop1),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(drop2),
            nn.Linear(256, num_classes)
            # ▶ Softmax not used here → CrossEntropyLoss includes LogSoftmax internally
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x


# Model Initialization
torch.manual_seed(SEED)
model = EMNISTConvNet(num_classes=NUM_CLASSES).to(device)

print(model)
print()

# Calculate number of parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameter    : {total_params:,}")
print(f"Trainable parameter: {trainable_params:,}")

NameError: name 'nn' is not defined

## Cell 7 – Dimension Flow Verification

In [14]:
# Check the output dimensions of each layer with dummy input.
model.eval()
dummy = torch.zeros(1, 1, 28, 28).to(device)

with torch.no_grad():
    d1 = model.conv_block1(dummy)
    d2 = model.conv_block2(d1)
    d3 = model.conv_block3(d2)
    d4 = model.classifier(d3)

print("Input          :", tuple(dummy.shape))
print("After Block 1  :", tuple(d1.shape),   "← 32 filter, 14×14")
print("After Block 2  :", tuple(d2.shape),   "← 64 filter, 7×7")
print("After Block 3  :", tuple(d3.shape),   "← 128 filter, 3×3")
print("After Flatten  : (1, 1152)",           "← 128×3×3 = 1152")
print("Output (logits):", tuple(d4.shape),   "← 26 kelas A–Z")

NameError: name 'model' is not defined

## Cell 8 – Loss Function, Optimizer and Scheduler

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4    # L2 light regularization
)

# StepLR: LR times 0.5 each 5 epoch
# Epoch  1– 5 : LR = 0.001
# Epoch  6–10 : LR = 0.0005
# Epoch 11–15 : LR = 0.00025
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Loss Function : CrossEntropyLoss")
print("Optimizer     : Adam (lr=0.001, weight_decay=1e-4)")
print("Scheduler     : StepLR (step_size=5, gamma=0.5)")
print(f"Device        : {device}")
print(f"Epochs        : {EPOCHS}")
print(f"Batch size    : {BATCH_SIZE}")

NameError: name 'nn' is not defined

## Cell 9 – Training Loop (with Progress Bar per Batch)

In [12]:
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss'  : [], 'val_acc'  : [],
    'lr'        : []
}

best_val_acc = 0.0
CHECKPOINT   = 'result/best_emnist_model.pth'

print(f"Mulai training di: {device}")
print(f"{'='*75}")

total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # TRAINING
    model.train()
    trn_loss, trn_correct, trn_total = 0.0, 0, 0

    pbar = tqdm(train_loader,
                desc=f'Epoch [{epoch:02d}/{EPOCHS}] TRAIN',
                leave=False, ncols=None)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        trn_loss    += loss.item() * images.size(0)
        _, preds     = torch.max(outputs, 1)
        trn_total   += labels.size(0)
        trn_correct += (preds == labels).sum().item()

        pbar.set_postfix(
            loss=f'{loss.item():.4f}',
            acc=f'{100.*trn_correct/trn_total:.1f}%'
        )

    avg_trn_loss = trn_loss / trn_total
    avg_trn_acc  = trn_correct / trn_total

    # VALIDATION
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item() * images.size(0)
            _, preds     = torch.max(outputs, 1)
            val_total   += labels.size(0)
            val_correct += (preds == labels).sum().item()

    avg_val_loss = val_loss / val_total
    avg_val_acc  = val_correct / val_total

    # Scheduler step (update learning rate)
    scheduler.step()
    cur_lr = scheduler.get_last_lr()[0]

    # Save history
    history['train_loss'].append(avg_trn_loss)
    history['train_acc'] .append(avg_trn_acc)
    history['val_loss']  .append(avg_val_loss)
    history['val_acc']   .append(avg_val_acc)
    history['lr']        .append(cur_lr)

    # Save best model
    marker = ''
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        marker = ' ✓ SAVED'

    epoch_time = time.time() - epoch_start
    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] "
        f"Train Loss: {avg_trn_loss:.4f}  Acc: {avg_trn_acc*100:.2f}% | "
        f"Val Loss: {avg_val_loss:.4f}  Acc: {avg_val_acc*100:.2f}% | "
        f"LR: {cur_lr:.6f} | {epoch_time:.0f}s{marker}"
    )

total_time = time.time() - total_start
print(f"{'='*75}")
print(f"Training finished! Total time: {total_time/60:.1f} minutes")
print(f"Best Validation Accuracy   : {best_val_acc*100:.2f}%")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared`.")

NameError: name 'device' is not defined

## Cell 10 – Load Best Model

In [11]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
print(f"Best model loaded from '{CHECKPOINT}'")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")

NameError: name 'model' is not defined

## Cell 11 – Loss and Accuracy Graph per Epoch (Seaborn)

In [10]:
epochs_range = list(range(1, EPOCHS + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History – EMNIST Letters Classification', fontsize=14, fontweight='bold')

# Plot 1: Loss
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'b-o', lw=2, ms=5, label='Train Loss')
ax.plot(epochs_range, history['val_loss'],   'r-o', lw=2, ms=5, label='Validation Loss')
ax.fill_between(epochs_range, history['train_loss'], history['val_loss'], alpha=0.08, color='purple')
ax.set_title('Loss per Epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.set_xticks(epochs_range[::2])

# Plot 2: Accuracy
ax = axes[1]
trn_acc_pct = [a*100 for a in history['train_acc']]
val_acc_pct = [a*100 for a in history['val_acc']]
ax.plot(epochs_range, trn_acc_pct, 'b-o', lw=2, ms=5, label='Train Accuracy')
ax.plot(epochs_range, val_acc_pct, 'r-o', lw=2, ms=5, label='Validation Accuracy')
ax.fill_between(epochs_range, trn_acc_pct, val_acc_pct, alpha=0.08, color='green')
# Tandai epoch terbaik
best_epoch = int(np.argmax(history['val_acc'])) + 1
ax.axvline(best_epoch, color='gold', linestyle='--', lw=1.5,
           label=f'Best epoch ({best_epoch})')
ax.set_title('Accuracy per Epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.set_xticks(epochs_range[::2])

# Plot 3: Learning Rate
ax = axes[2]
ax.plot(epochs_range, history['lr'], 's-', color='darkorange', lw=2, ms=6)
ax.fill_between(epochs_range, history['lr'], alpha=0.15, color='darkorange')
ax.set_title('Learning Rate Decay (StepLR)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_xticks(epochs_range[::2])
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.6f'))

plt.tight_layout()
plt.savefig('result/plot_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: result/plot_training_history.png")

NameError: name 'EPOCHS' is not defined

## Cell 12 – Full Evaluation on Test Set

In [9]:
model.eval()
all_preds  = []
all_labels = []
all_probs  = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluating Test Set'):
        images = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds .extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs .extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

overall_acc = (all_preds == all_labels).mean() * 100
n_correct   = (all_preds == all_labels).sum()
n_total     = len(all_labels)

print(f"\n{'='*55}")
print(f"  TEST SET EVALUATION RESULTS")
print(f"{'='*55}")
print(f"  Total test samples : {n_total:,}")
print(f"  Correctly classified: {n_correct:,}")
print(f"  Incorrectly classified: {n_total - n_correct:,}")
print(f"  Overall Accuracy   : {overall_acc:.2f}%")
print(f"{'='*55}")

NameError: name 'model' is not defined

## Cell 13 – Classification Report (Per-Class Precision, Recall, F1)

In [8]:
report = classification_report(
    all_labels, all_preds,
    target_names=ALPHABET, digits=4
)
print("\n" + "="*60)
print("  CLASSIFICATION REPORT (per class)")
print("="*60)
print(report)

# Showed as DataFrame for better readability and to easily identify best/worst classes
report_dict = classification_report(
    all_labels, all_preds, target_names=ALPHABET, output_dict=True
)
df_report = pd.DataFrame(report_dict).T.round(4)
print("\n5 Classes with Highest F1-Score:")
print(df_report.iloc[:26].sort_values('f1-score', ascending=False).head(5))
print("\n5 Classes with Lowest F1-Score:")
print(df_report.iloc[:26].sort_values('f1-score', ascending=True).head(5))

NameError: name 'classification_report' is not defined

## Cell 14 – Confusion Matrix Heatmap (Seaborn)

In [7]:
cm            = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Confusion Matrix – EMNIST Letter Classification',
             fontsize=14, fontweight='bold')

# Raw Counts
sns.heatmap(cm, ax=axes[0],
            annot=True, fmt='d', cmap='Blues',
            xticklabels=ALPHABET, yticklabels=ALPHABET,
            annot_kws={'size': 6.5}, linewidths=0.2, linecolor='#ecf0f1')
axes[0].set_title('Number of Predictions (Raw Counts)', fontsize=12)
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label')
axes[0].tick_params(axis='both', labelsize=9)

# Normalized (%)
sns.heatmap(cm_normalized, ax=axes[1],
            annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=ALPHABET, yticklabels=ALPHABET,
            annot_kws={'size': 6.5}, linewidths=0.2, linecolor='#ecf0f1',
            vmin=0, vmax=100)
axes[1].set_title('Normalized per Row (%)', fontsize=12)
axes[1].set_xlabel('Predicted Label'); axes[1].set_ylabel('True Label')
axes[1].tick_params(axis='both', labelsize=9)

plt.tight_layout()
plt.savefig('result/plot_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: result/plot_confusion_matrix.png")

NameError: name 'confusion_matrix' is not defined

## Cell 15 – Accuracy Per Class (Bar Chart + Threshold)

In [6]:
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

colors = [
    '#27ae60' if acc >= 90   # Green: ≥ 90%
    else '#f39c12' if acc >= 80  # Orange: 80–90%
    else '#e74c3c'               # Red: < 80%
    for acc in per_class_acc
]

fig, ax = plt.subplots(figsize=(17, 6))
bars = ax.bar(ALPHABET, per_class_acc, color=colors, edgecolor='white', linewidth=0.6)

# Label nilai di atas bar
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f'{acc:.1f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

# Garis referensi
ax.axhline(overall_acc, color='navy',   linestyle='--', lw=2.0,
           label=f'Overall Acc: {overall_acc:.1f}%')
ax.axhline(90,          color='#27ae60', linestyle=':',  lw=1.5, alpha=0.8, label='90%')
ax.axhline(80,          color='#f39c12', linestyle=':',  lw=1.5, alpha=0.8, label='80%')

# Legend warna kelas
legend_patches = [
    mpatches.Patch(color='#27ae60', label='≥ 90% (Very Good)'),
    mpatches.Patch(color='#f39c12', label='80–90% (Good)'),
    mpatches.Patch(color='#e74c3c', label='< 80% (Needs Attention)'),
]
ax.legend(handles=legend_patches + [
    plt.Line2D([0],[0], color='navy',  linestyle='--', lw=2, label=f'Overall: {overall_acc:.1f}%'),
], fontsize=10, loc='lower right')

ax.set_xlabel('Letter'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Per Class Accuracy on Test Set')
ax.set_ylim(0, 115); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('result/plot_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n  5 Letters Most Easily Recognized:")
top5 = sorted(zip(ALPHABET, per_class_acc), key=lambda x: -x[1])[:5]
for ltr, acc in top5: print(f"    {ltr}: {acc:.2f}%")

print(f"\n  5 Letters Most Frequently Misclassified:")
bot5 = sorted(zip(ALPHABET, per_class_acc), key=lambda x: x[1])[:5]
for ltr, acc in bot5: print(f"    {ltr}: {acc:.2f}%")

NameError: name 'cm' is not defined

## Cell 16 – Visualization of Test Sample Predictions (True vs False)

In [5]:
def show_predictions(model, test_data, device, alphabet, n_show=25, only_wrong=False):
    model.eval()
    images_list, labels_list, preds_list, confs_list = [], [], [], []

    # Collect samples
    with torch.no_grad():
        for imgs, lbls in test_loader:
            outs  = model(imgs.to(device))
            probs = torch.softmax(outs, dim=1)
            prd, conf_idx = torch.max(probs, 1)

            for i in range(len(lbls)):
                correct = (conf_idx[i].item() == lbls[i].item())
                if only_wrong and correct:
                    continue
                images_list.append(imgs[i].squeeze().numpy())
                labels_list.append(lbls[i].item())
                preds_list .append(conf_idx[i].item())
                confs_list .append(probs[i, conf_idx[i]].item() * 100)
                if len(images_list) >= n_show:
                    break
            if len(images_list) >= n_show: break

    # Plot
    cols = 5; rows = (n_show + cols - 1) // cols
    title_str = ('Wrong Predictions' if only_wrong else 'Sample Predictions')
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    fig.suptitle(title_str, fontsize=14, fontweight='bold')

    for i in range(rows * cols):
        row, col = i // cols, i % cols
        ax = axes[row, col]
        if i >= len(images_list):
            ax.set_visible(False); continue

        img  = images_list[i]
        true = labels_list[i]
        pred = preds_list[i]
        conf = confs_list[i]
        ok   = (pred == true)

        ax.imshow(img, cmap='gray', interpolation='bilinear')
        color = '#27ae60' if ok else '#e74c3c'
        ax.set_title(
            f"True: {alphabet[true]}\nPred: {alphabet[pred]} ({conf:.0f}%)",
            fontsize=9, color=color, fontweight='bold'
        )
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(2.5)

    plt.tight_layout()
    fname = 'result/plot_wrong_predictions.png' if only_wrong else 'result/plot_sample_predictions.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved: {fname}")


# Showed 25 random right predictions from the test set
show_predictions(model, test_data, device, ALPHABET, n_show=25, only_wrong=False)

NameError: name 'model' is not defined

## Cell 17 – WRONG Prediction (Error Analysis)

In [4]:
# Show 20 examples of WRONG predictions for analysis
show_predictions(model, test_data, device, ALPHABET, n_show=20, only_wrong=True)

NameError: name 'show_predictions' is not defined

## Cell 18 – Single Image Prediction (Single Inference)

In [3]:
def predict_single(index: int, dataset=test_data):
    model.eval()
    img, true_label = dataset[index]

    with torch.no_grad():
        output = model(img.unsqueeze(0).to(device))
        probs  = torch.softmax(output, dim=1).cpu().numpy()[0]
        pred   = probs.argmax()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4),
                             gridspec_kw={'width_ratios': [1, 3]})

    # Image
    axes[0].imshow(img.squeeze().numpy(), cmap='gray', interpolation='bilinear')
    correct = (pred == true_label)
    color   = '#27ae60' if correct else '#e74c3c'
    axes[0].set_title(
        f"True: {ALPHABET[true_label]}  |  Pred: {ALPHABET[pred]}\n"
        f"Confidence: {probs[pred]*100:.2f}%",
        fontsize=12, color=color, fontweight='bold'
    )
    axes[0].axis('off')

    # Probability
    bar_colors = [
        '#27ae60' if i == true_label else
        '#e74c3c' if i == pred else
        '#3498db'
        for i in range(NUM_CLASSES)
    ]
    bars = axes[1].barh(ALPHABET, probs * 100, color=bar_colors, edgecolor='white')
    axes[1].set_xlabel('Probability (%)')
    axes[1].set_title('Probability Distribution over Classes', fontsize=12)
    axes[1].set_xlim(0, 110)
    axes[1].invert_yaxis()
    for bar, prob in zip(bars, probs):
        if prob > 0.005:
            axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                         f'{prob*100:.2f}%', va='center', fontsize=8)

    legend_p = [
        mpatches.Patch(color='#27ae60', label=f'True label ({ALPHABET[true_label]})'),
        mpatches.Patch(color='#e74c3c', label=f'Predicted ({ALPHABET[pred]})'),
        mpatches.Patch(color='#3498db', label='Other classes'),
    ]
    axes[1].legend(handles=legend_p, fontsize=9, loc='lower right')

    plt.tight_layout()
    plt.show()

# Example: Predict a single sample from the test set by index
predict_single(42)    # change the number to select a different sample (0 – 14799)
predict_single(1234)  # another example
predict_single(5678)  # another example

NameError: name 'test_data' is not defined

## Cell 19 – Interactive Viewer with PyQt6 (Optional)

> **How ​​to use PyQt6 in Jupyter:**
> 1. Restart the kernel
> 2. In the first cell, replace `%matplotlib inline` → `%matplotlib qt6`
> 3. The plot will open in a separate Qt6 window that can be zoomed, panned, and saved.
>
> Or run the script below to open the interactive PyQt6 Prediction Viewer.
> Outside of Jupyter, use `app.exec()` (uncomment the last line).

In [2]:
try:
    from PyQt6.QtWidgets import (
        QApplication, QMainWindow, QWidget,
        QVBoxLayout, QHBoxLayout, QLabel,
        QPushButton, QSpinBox, QGroupBox
    )
    from PyQt6.QtCore import Qt
    from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
    from matplotlib.figure import Figure
    import matplotlib
    matplotlib.use('QtAgg')
    PYQT6_OK = True
    print("PyQt6 detected ✓ Interactive Viewer is ready to launch.")
except ImportError:
    PYQT6_OK = False
    print("PyQt6 not found. Install: pip install PyQt6")


if PYQT6_OK:
    class PredictionViewer(QMainWindow):
        """Interactive window for exploring EMNIST model predictions."""

        def __init__(self, model, dataset, device, alphabet):
            super().__init__()
            self.model   = model
            self.dataset = dataset
            self.device  = device
            self.alpha   = alphabet
            self.setWindowTitle('EMNIST Letter Prediction Viewer')
            self.setMinimumSize(860, 540)
            self._build_ui()
            self._predict(0)

        def _build_ui(self):
            central = QWidget()
            self.setCentralWidget(central)
            main = QHBoxLayout(central)

            # Left panel
            left = QVBoxLayout()

            nav = QGroupBox('Navigator')
            nav_lay = QHBoxLayout()
            nav_lay.addWidget(QLabel('Index:'))
            self.spin = QSpinBox()
            self.spin.setRange(0, len(self.dataset) - 1)
            self.spin.valueChanged.connect(self._predict)
            nav_lay.addWidget(self.spin)
            btn_p = QPushButton('◀ Prev')
            btn_n = QPushButton('Next ▶')
            btn_p.clicked.connect(lambda: self.spin.setValue(self.spin.value() - 1))
            btn_n.clicked.connect(lambda: self.spin.setValue(self.spin.value() + 1))
            nav_lay.addWidget(btn_p)
            nav_lay.addWidget(btn_n)
            nav.setLayout(nav_lay)
            left.addWidget(nav)

            self.fig_img = Figure(figsize=(3, 3), facecolor='white')
            self.canvas_img = FigureCanvas(self.fig_img)
            self.ax_img = self.fig_img.add_subplot(111)
            left.addWidget(self.canvas_img)

            self.lbl_pred = QLabel('Prediction: -')
            self.lbl_pred.setAlignment(Qt.AlignmentFlag.AlignCenter)
            self.lbl_pred.setStyleSheet('font-size:20px; font-weight:bold;')
            left.addWidget(self.lbl_pred)

            self.lbl_true = QLabel('True Label: -')
            self.lbl_true.setAlignment(Qt.AlignmentFlag.AlignCenter)
            self.lbl_true.setStyleSheet('font-size:14px;')
            left.addWidget(self.lbl_true)

            main.addLayout(left, 1)

            # Right panel: bar chart of probabilities
            self.fig_bar = Figure(figsize=(5, 7), facecolor='white')
            self.canvas_bar = FigureCanvas(self.fig_bar)
            self.ax_bar = self.fig_bar.add_subplot(111)
            main.addWidget(self.canvas_bar, 2)

        def _predict(self, idx):
            img, true_lbl = self.dataset[idx]
            self.model.eval()
            with torch.no_grad():
                out   = self.model(img.unsqueeze(0).to(self.device))
                probs = torch.softmax(out, dim=1).cpu().numpy()[0]
            pred = probs.argmax()

            # Image
            self.ax_img.clear()
            self.ax_img.imshow(img.squeeze().numpy(), cmap='gray')
            self.ax_img.axis('off')
            self.canvas_img.draw()

            # Label
            ok    = (pred == true_lbl)
            color = 'green' if ok else 'red'
            self.lbl_pred.setText(f'Pred: {self.alpha[pred]} ({probs[pred]*100:.1f}%)')
            self.lbl_pred.setStyleSheet(f'font-size:20px; font-weight:bold; color:{color};')
            self.lbl_true.setText(f'True: {self.alpha[true_lbl]}')

            # Bar chart
            self.ax_bar.clear()
            clrs = ['#27ae60' if i == true_lbl
                    else '#e74c3c' if i == pred
                    else '#3498db' for i in range(26)]
            self.ax_bar.barh(self.alpha, probs * 100, color=clrs)
            self.ax_bar.set_xlabel('Probability (%)')
            self.ax_bar.set_title('Class Probabilities')
            self.ax_bar.set_xlim(0, 115)
            self.ax_bar.invert_yaxis()
            self.fig_bar.tight_layout()
            self.canvas_bar.draw()

    app = QApplication.instance() or QApplication(sys.argv)
    viewer = PredictionViewer(model, test_data, device, ALPHABET)
    viewer.show()
    # app.exec()   # ← Uncomment if not in Jupyter (standalone)

PyQt6 detected ✓ Interactive Viewer is ready to launch.


NameError: name 'sys' is not defined

## Cell 20 – Save and Load Model

In [1]:
# Save full model (architecture + weight)
torch.save({
    'model_state_dict' : model.state_dict(),
    'optimizer_state'  : optimizer.state_dict(),
    'history'          : history,
    'best_val_acc'     : best_val_acc,
    'epochs'           : EPOCHS,
    'num_classes'      : NUM_CLASSES,
    'alphabet'         : ALPHABET,
}, 'result/emnist_final_checkpoint.pth')
print("Model saved: result/emnist_final_checkpoint.pth")

# How to reload the model later:
# checkpoint = torch.load('emnist_final_checkpoint.pth', map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# print(f"Loaded. Best val acc: {checkpoint['best_val_acc']*100:.2f}%")

NameError: name 'torch' is not defined

---
## Final Summary

| Components | Details |
|---|---|
| Dataset | EMNIST Letters (A–Z, 26 classes) |
| Training set | 88,800 samples |
| Test set | 14,800 samples |
| Architecture | 3× ConvBlock (BatchNorm) + 3× FC (Dropout) |
| Parameters | ~1.5 million |
| Device | CUDA GPU (RTX 4050) |
| Batch size | 128 |
| Epochs | 15 |
| Optimizer | Adam + StepLR scheduler |
| Visualization | Seaborn heatmap + per-class bar chart |
| Interactive | PyQt6 Prediction Viewer |